In [ ]:
# Data file: set CMSOPEN_DATA_FILE to the absolute path of a NanoAOD ROOT file
# before starting Jupyter, e.g. the 2012C DoubleMuParked file described in DATA.md.
import os
from pathlib import Path

if not os.environ.get("CMSOPEN_DATA_FILE"):
    raise RuntimeError(
        "CMSOPEN_DATA_FILE is not set. Start Jupyter with "
        "CMSOPEN_DATA_FILE=/absolute/path/to/file.root jupyter lab ..."
    )
DATA_FILE = Path(os.environ["CMSOPEN_DATA_FILE"])
if not DATA_FILE.exists():
    raise FileNotFoundError(f"CMSOPEN_DATA_FILE points to a missing file: {DATA_FILE}")
print(f"Data file: {DATA_FILE}")

# Layer 3: selecting Z muons with `cmsopen.selection`

The raw muon collection is mostly background: soft muons from hadron decays, muons inside jets, and badly reconstructed candidates. Each cut removes one kind:

| cut | removes |
|---|---|
| dimuon trigger (`HLT_Mu17_Mu8` OR `HLT_Mu17_TkMu8`) | events outside a well-understood trigger |
| $p_T > 20$ GeV | soft muons from decays and noise; keeps the trigger on its plateau |
| $\lvert\eta\rvert < 2.4$ | candidates outside the muon chambers and tracker |
| tight ID | punch-through hadrons, decays in flight, poorly measured tracks |
| relative isolation $< 0.12$ | muons from b and c decays inside jets |
| at least 2 muons | events that cannot contain a $Z\to\mu^+\mu^-$ candidate |

All cuts are strict: a value exactly at the threshold fails. A `Selection` is immutable and chainable, and it records a *cutflow*: how many muons and events survive each step.

In [ ]:
import awkward as ak
import matplotlib.pyplot as plt
import numpy as np
import uproot

import cmsopen
from cmsopen import invariant_mass, open_events, select_muons
from cmsopen.ingestion import DIMUON_TRIGGERS_2012

print(f"cmsopen imported from: {cmsopen.__file__}")

## The cutflow over all events in the file

`prev` is the efficiency relative to the previous row (what that cut alone removed), `start` is relative to row 0. Particle cuts remove muons but not events, so the last column only moves at the trigger and at `require(>= 2)`.

In [ ]:
PT_MIN = 20.0  # GeV
ETA_MAX = 2.4
ISO_MAX = 0.12

with open_events(DATA_FILE) as events:
    everything = select_muons(events)
    triggered = everything.require_trigger(*DIMUON_TRIGGERS_2012)
# The particle cuts below are cmsopen.ZMUMU_MUON_CUTS, written out step by step.
final = (
    triggered
    .min_pt(PT_MIN)
    .max_eta(ETA_MAX)
    .quality_id("tight")
    .max_rel_iso(ISO_MAX)
    .require(2)
)
cutflow = final.cutflow()
print(repr(final))
print()
print(cutflow)

## The junk is gone

The raw collection has candidates far outside the detector ($|\eta|$ near 12) and with unphysical momenta. After the selection every muon sits inside the acceptance and above threshold.

In [ ]:
raw, sel = everything.particles, final.particles
raw_pt, raw_eta = ak.flatten(raw.p4.pt), ak.flatten(raw.p4.eta)
sel_pt, sel_eta = ak.flatten(sel.p4.pt), ak.flatten(sel.p4.eta)
print(f"  {'':12}{'muons':>12}{'max |eta|':>12}{'max pt [GeV]':>15}{'min pt [GeV]':>15}")
for label, pt, eta in (("before", raw_pt, raw_eta), ("after", sel_pt, sel_eta)):
    print(f"  {label:12}{len(pt):>12,}{float(ak.max(abs(eta))):>12.4f}{float(ak.max(pt)):>15.1f}{float(ak.min(pt)):>15.4f}")

## Dimuon mass: full selection vs the crude layer 5 selection

In each selected event we take the two highest-$p_T$ surviving muons and keep them if their charges are opposite. (Proper pairing is layer 4; here it is a plain awkward mask.) The *crude* selection is the one from the layer 5 demo: the two leading muons of all muons, opposite charge, $p_T > 20$ GeV and $|\eta| < 2.4$, with no trigger, ID or isolation.

In [ ]:
def leading_pair_mass(muons):
    """Invariant mass of the two highest-pT muons in each event with opposite charges."""
    muons = muons[ak.argsort(muons.p4.pt, axis=1, ascending=False)]
    muons = muons[muons.num >= 2]
    lead, sub = muons[:, 0], muons[:, 1]
    opposite = ak.to_numpy(lead.charge * sub.charge < 0)
    return ak.to_numpy(invariant_mass(lead[opposite].p4, sub[opposite].p4))


def crude_layer5_mass(muons):
    """The layer 5 demo selection: leading two muons, opposite charge, pt > 20 GeV, |eta| < 2.4."""
    muons = muons[ak.argsort(muons.p4.pt, axis=1, ascending=False)]
    muons = muons[muons.num >= 2]
    lead, sub = muons[:, 0], muons[:, 1]
    selected = ak.to_numpy(
        (lead.charge * sub.charge < 0)
        & (lead.p4.pt > PT_MIN) & (sub.p4.pt > PT_MIN)
        & (np.abs(lead.p4.eta) < ETA_MAX) & (np.abs(sub.p4.eta) < ETA_MAX)
    )
    return ak.to_numpy(invariant_mass(lead[selected].p4, sub[selected].p4))


crude = crude_layer5_mass(raw)
selected = leading_pair_mass(sel)
print(f"crude layer 5 selection:   {len(crude):,} opposite-charge leading pairs")
print(f"full layer 3 selection:    {len(selected):,} opposite-charge leading pairs "
      f"(of {final.n_events:,} events with >= 2 selected muons)")

BINS = np.arange(60.0, 122.0, 2.0)
c_counts, edges = np.histogram(crude, bins=BINS)
s_counts, _ = np.histogram(selected, bins=BINS)

The upper panel shows both spectra as step histograms. The lower panel is the ratio of the two counts in each bin. (It is not strictly a survival fraction: the full selection picks its leading pair after the cuts, so its pairs are not always a subset of the crude ones.) The ratio is highest under the Z peak and lower in the sidebands, because the trigger, ID and isolation remove background pairs more than Z pairs.

In [ ]:
fig, (ax, rax) = plt.subplots(
    2, 1, figsize=(7.5, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05}
)
ax.stairs(c_counts, edges, linewidth=1.5, color="tab:gray", label=f"crude layer 5 selection ({int(c_counts.sum()):,} pairs)")
ax.stairs(s_counts, edges, linewidth=1.8, color="tab:blue", label=f"full layer 3 selection ({int(s_counts.sum()):,} pairs)")
ax.axvline(91.1876, color="tab:red", linestyle="--", linewidth=1, label="PDG Z mass 91.1876 GeV")
ax.set_ylabel("Opposite-charge pairs / 2 GeV")
ax.set_xlim(edges[0], edges[-1])
ax.set_ylim(0, 1.3 * max(c_counts.max(), s_counts.max()))
ax.legend(loc="upper right", fontsize=9, frameon=False)
ax.text(0.99, 1.01, "CMS Open Data, 2012C DoubleMuParked, 8 TeV", transform=ax.transAxes, ha="right", va="bottom", fontsize=9)
with np.errstate(divide="ignore", invalid="ignore"):
    ratio = np.where(c_counts > 0, s_counts / c_counts, np.nan)
rax.stairs(ratio, edges, baseline=None, color="tab:blue", linewidth=1.5)
rax.axvline(91.1876, color="tab:red", linestyle="--", linewidth=1)
rax.set_ylabel("full / crude")
rax.set_ylim(0, 1)
rax.set_xlabel("Dimuon invariant mass $M_{\\mu\\mu}$ [GeV]")
plt.show()

## Self-checks

1. The cutflow never increases from one row to the next, in any of its three counts.
2. Every cutflow number equals an independent recomputation that reads the branches with uproot and applies the cuts with plain awkward, without calling any cmsopen cut.
3. After the cuts, max $|\eta| < 2.4$ and min $p_T > 20$ GeV.
4. After selection, the highest bin between 70 and 110 GeV lies within 88 to 94 GeV.
5. Peak fraction: the fraction of 60-120 GeV pairs that lie in 85-97 GeV is higher after the full selection than for the crude one. This is a rough measure of how concentrated the spectrum is around the Z, not a true purity, which would need the background under the peak to be modelled and subtracted.

In [ ]:
class Checks:
    """Collects self-check results and prints one OK/FAIL/SKIP line each."""

    def __init__(self):
        self.failures = []

    def check(self, name, ok, detail):
        print(f"{'OK' if ok else 'FAIL'}: {name} ({detail})")
        if not ok:
            self.failures.append(name)

    def skip(self, name, reason):
        print(f"SKIP: {name} ({reason})")

In [ ]:
PEAK_SEARCH = (70.0, 110.0)
PEAK_WINDOW = (88.0, 94.0)
WIDE = (60.0, 120.0)
CORE = (85.0, 97.0)


def independent_cutflow(path):
    """Recompute every cutflow row with uproot and plain awkward, without cmsopen."""
    with uproot.open(path) as f:
        a = f["Events"].arrays(
            ["Muon_pt", "Muon_eta", "Muon_tightId", "Muon_pfRelIso04_all", *DIMUON_TRIGGERS_2012]
        )
    fired = np.logical_or.reduce([ak.to_numpy(a[name]) for name in DIMUON_TRIGGERS_2012])

    def row(mask, events):
        per_event = ak.sum(mask, axis=1)[events]
        return int(ak.sum(per_event)), int(ak.sum(per_event > 0)), int(np.sum(events))

    everything = ak.ones_like(a.Muon_pt, dtype=bool)
    m_pt = a.Muon_pt > PT_MIN
    m_eta = m_pt & (abs(a.Muon_eta) < ETA_MAX)
    m_id = m_eta & a.Muon_tightId
    m_iso = m_id & (a.Muon_pfRelIso04_all < ISO_MAX)
    two = fired & ak.to_numpy(ak.sum(m_iso, axis=1) >= 2)
    all_events = np.ones(len(a), dtype=bool)
    return [
        row(everything, all_events),
        row(everything, fired),
        row(m_pt, fired),
        row(m_eta, fired),
        row(m_id, fired),
        row(m_iso, fired),
        row(m_iso, two),
    ]


def fraction_in(mass, inner, outer):
    n_outer = int(np.sum((mass >= outer[0]) & (mass < outer[1])))
    n_inner = int(np.sum((mass >= inner[0]) & (mass < inner[1])))
    return (n_inner / n_outer if n_outer else float("nan")), n_inner, n_outer


checks = Checks()
d = cutflow.to_dict()
columns = ("n_particles", "n_events_with_particles", "n_events")
increases = [(c, i) for c in columns for i in range(1, len(cutflow)) if d[c][i] > d[c][i - 1]]
checks.check(
    "cutflow is monotonically non-increasing in particles, events with >= 1 particle and events",
    not increases,
    f"{len(cutflow)} rows; increases found: {increases}" if increases else f"{len(cutflow)} rows x 3 columns",
)

ours = [tuple(d[c][i] for c in columns) for i in range(len(cutflow))]
theirs = independent_cutflow(DATA_FILE)
pairs = "; ".join(f"{d['cut'][i]}: {o[0]:,}/{o[1]:,}/{o[2]:,} vs {t[0]:,}/{t[1]:,}/{t[2]:,}"
                  for i, (o, t) in enumerate(zip(ours, theirs)))
checks.check(
    "cutflow agrees with an independent uproot + plain awkward recomputation",
    ours == theirs,
    f"particles/events>=1/events, cmsopen vs independent: {pairs}",
)

max_abs_eta, min_pt = float(ak.max(abs(sel_eta))), float(ak.min(sel_pt))
checks.check(
    f"after the cuts: max |eta| < {ETA_MAX} and min pt > {PT_MIN:.0f} GeV",
    max_abs_eta < ETA_MAX and min_pt > PT_MIN,
    f"max |eta| = {max_abs_eta:.4f}, min pt = {min_pt:.4f} GeV over {len(sel_pt):,} muons",
)

in_search = (edges[:-1] >= PEAK_SEARCH[0]) & (edges[1:] <= PEAK_SEARCH[1])
peak = int(np.flatnonzero(in_search)[np.argmax(s_counts[in_search])])
peak_lo, peak_hi = edges[peak], edges[peak + 1]
checks.check(
    f"Z peak after selection: highest bin in {PEAK_SEARCH[0]:.0f}-{PEAK_SEARCH[1]:.0f} GeV lies within "
    f"{PEAK_WINDOW[0]:.0f}-{PEAK_WINDOW[1]:.0f} GeV",
    bool(peak_lo >= PEAK_WINDOW[0] and peak_hi <= PEAK_WINDOW[1]),
    f"peak bin {peak_lo:.0f}-{peak_hi:.0f} GeV with {s_counts[peak]:,} pairs",
)

f_sel, n_core_s, n_wide_s = fraction_in(selected, CORE, WIDE)
f_crude, n_core_c, n_wide_c = fraction_in(crude, CORE, WIDE)
checks.check(
    f"peak fraction: fraction of {WIDE[0]:.0f}-{WIDE[1]:.0f} GeV pairs inside {CORE[0]:.0f}-{CORE[1]:.0f} GeV "
    "is higher after the full selection than for the crude selection",
    f_sel > f_crude,
    f"full {f_sel:.4f} ({n_core_s:,}/{n_wide_s:,}) vs crude {f_crude:.4f} ({n_core_c:,}/{n_wide_c:,})",
)

if checks.failures:
    raise AssertionError(f"{len(checks.failures)} check(s) failed: {', '.join(checks.failures)}")
print("All checks passed")